In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

ROOT = Path.cwd().parent

env_path = ROOT / ".env.backtest"

load_dotenv(env_path)

print(env_path)
print(os.getenv("MONGO_HOST"))

os.chdir(Path.cwd().parent)

/Users/paulvogt/mongo_db_streamlit/.env.backtest
localhost


In [2]:
%pwd

'/Users/paulvogt/mongo_db_streamlit'

In [3]:
from infrastructure.mongo.mongo_repository import MongoRepository
from config.mongo_config import MongoCollection, MongoDatabase, MongoUser
from infrastructure.mongo.mongo_connection import MongoConnection
from core.application.notebook_service import NotebookService

In [4]:
with MongoConnection(user=MongoUser.DASHBOARDUSER) as conn:
    finance_repo = MongoRepository(conn, MongoDatabase.PROCESSED, MongoCollection.FINANCEDATA)
    company_repo = MongoRepository(conn, MongoDatabase.PROCESSED, MongoCollection.COMPANYDATA)
    sector_repo = MongoRepository(conn,  MongoDatabase.PROCESSED, MongoCollection.SECTORDATA)
    sp_500_repo = MongoRepository(conn,  MongoDatabase.PROCESSED, MongoCollection.SP500)
    constituents_repo = MongoRepository(conn,  MongoDatabase.PROCESSED, MongoCollection.SCD_CONSTITUENTS)
    
    notebook_service = NotebookService(finance_repo, company_repo, sector_repo, sp_500_repo, constituents_repo)

    df = notebook_service.run()
    

/Users/paulvogt/mongo_db_streamlit/core/application/notebook_service.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_result = pd.concat([df_result, df_grouped[res_cols]])
/Users/paulvogt/mongo_db_streamlit/core/application/notebook_service.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_momentum, df_value]).reset_index(drop=True)
/Users/paulvogt/mongo_db_streamlit/core/application/notebook_service.py:61: FutureWarning: The behavior of DataFrame concatenation

In [5]:
for (sector, strategy), group in df.groupby(["sector", "strategy"]):

    triangle = group.pivot_table(
        index="buyyear",
        columns="sellyear",
        values="rendite"
    ).sort_index().sort_index(axis=1)

    print("\n" + "="*60)
    print(f"{sector} | {strategy}")
    print("="*60)

    print(triangle.round(2))


Basic Materials | momentum_score
sellyear   2011   2012   2013   2014   2015   2016   2017   2018   2019  \
buyyear                                                                   
2010     -11.23   4.85  11.60  10.61   5.37   7.07  10.67   6.77   8.64   
2011        NaN  27.58  25.51  20.86  13.74  14.07  16.16  12.56  15.58   
2012        NaN    NaN  28.21  17.05  10.14  13.30  16.12   9.73  12.13   
2013        NaN    NaN    NaN   0.10  -9.15   1.97   9.44   1.81   3.84   
2014        NaN    NaN    NaN    NaN  -3.95   4.77  11.96   5.68   9.90   
2015        NaN    NaN    NaN    NaN    NaN  24.29  25.89   9.25  13.54   
2016        NaN    NaN    NaN    NaN    NaN    NaN  25.22   4.64  10.59   
2017        NaN    NaN    NaN    NaN    NaN    NaN    NaN -10.69   7.97   
2018        NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN  27.88   
2019        NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
2020        NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN  

In [6]:
df.to_csv("backtest/data/backtest_data.csv", index=False)